In [23]:
!pip install pandas matplotlib numpy

In [24]:
import ast
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def safe_parse_json(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            return json.loads(value)
        except (TypeError, json.JSONDecodeError):
            return ast.literal_eval(value)
    return []


def get_people_names(row, column, limit=10):
    people = safe_parse_json(row[column])
    names = []
    for item in people[:limit]:
        if isinstance(item, dict) and item.get("name"):
            names.append(item["name"])
    return names


def build_feature_matrix(df):
    vocab = {}
    movie_vectors = []

    for _, row in df.iterrows():
        names = get_people_names(row, "cast", limit=8) + get_people_names(row, "crew", limit=8)
        for name in names:
            if name not in vocab:
                vocab[name] = len(vocab)

    for _, row in df.iterrows():
        names = get_people_names(row, "cast", limit=8) + get_people_names(row, "crew", limit=8)
        vector = np.zeros(len(vocab), dtype=np.float32)
        for name in set(names):
            if name in vocab:
                vector[vocab[name]] = 1.0
        movie_vectors.append(vector)

    return vocab, np.vstack(movie_vectors)


def recommend_movies(title, df, movie_vectors, top_n=10):
    if title not in df["title"].values:
        raise ValueError(f"Movie '{title}' not found in the dataset.")

    idx = df.index[df["title"] == title][0]
    movie_second_vectors = movie_vectors[idx]

    scores = movie_vectors @ movie_second_vectors
    scores[idx] = -1

    top_indices = np.argsort(scores)[::-1][:top_n]
    recommended = df.iloc[top_indices][["title", "movie_id"]].copy()
    recommended["score"] = scores[top_indices]
    return recommended.reset_index(drop=True)


In [ ]:
df = pd.read_csv("data/tmdb_5000_credits.csv")
vocab, movie_vectors = build_feature_matrix(df)

# Example
recommended = recommend_movies("Avatar", df, movie_vectors, top_n=10)
recommended